# Diffusion inside a sphere
The model<sup>1,2</sup> describes a particle that diffuses with diffusion coefficient $D$ inside a sphere of radius $R$ that it cannot leave, for example an atom of a side chain moving within a limited volume. The sphere is isotropic, so the scattering depends only on the magnitude of $Q$:

$$ S(Q, E) = A_0(Q)\,\delta(E) + \sum_{(l, n)} A_{nl}(Q)\, L(E, \Gamma_{nl}) $$

with

$$ A_0(Q) = \left(\frac{3 j_1(QR)}{QR}\right)^2,\quad
A_{nl}(Q) = (2l+1) \frac{6 x_{nl}^2}{x_{nl}^2 - l(l+1)}
\left(\frac{QR\, j_{l+1}(QR) - l\, j_l(QR)}{(QR)^2 - x_{nl}^2}\right)^2,\quad
\Gamma_{nl} = \frac{\hbar x_{nl}^2 D}{R^2}, $$

where $j_l$ are the spherical Bessel functions and $x_{nl}$ the roots of $j_l'(x) = 0$, which express that no particles flow through the surface. $A_0$ is the Elastic Incoherent Structure Factor (EISF) and $A_{nl}$ the Quasi-Elastic Incoherent Structure Factor (QISF) of the root $x_{nl}$. The widths do not depend on $Q$; only the QISF do.

The sum is infinite. The model uses the 98 roots below 26.84, the same roots as Mantid<sup>3</sup>, and by default only the lowest 10 of them, because every term adds Parameters that slow down a fit.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np

import easydynamics as edyn

%matplotlib widget

## The model at different Q values

In [ ]:
# Q is in Angstrom^-1, the radius in Angstrom and D in Angstrom^2/ps.
# With R = 1 Angstrom, QR stays below 1.5, where 10 terms are enough.
Q = np.linspace(0.3, 1.5, 6)

sphere_model = edyn.DiffusionInSphere(
    scale=1.0,
    radius=1.0,
    diffusion_coefficient=0.1,
    Q=Q,
)
print(sphere_model)

In [ ]:
# Plot the quasi-elastic part (all Lorentzians) at every Q. The delta function
# would be a spike at E = 0; its weight, the EISF, is plotted further down.
energy = np.linspace(-0.5, 0.5, 501)
component_collections = sphere_model.get_component_collections()

cmap = plt.cm.jet
nQ = len(component_collections)
plt.figure()
for Q_index, collection in enumerate(component_collections):
    quasielastic = sum(
        component.evaluate(energy)
        for component in collection
        if isinstance(component, edyn.Lorentzian)
    )
    color = cmap(Q_index / (nQ - 1))
    plt.plot(energy, quasielastic, label=f'Q={Q[Q_index]:.2f} Å^-1', color=color)

plt.legend()
plt.xlabel('Energy (meV)')
plt.ylabel('Intensity (arb. units)')
plt.title('Diffusion inside a sphere, R = 1 Å')
plt.show()

## Roots and widths

The Lorentzian of order $j$ belongs to the $j$-th smallest root. `roots` gives each root as $(x_{nl}, l, n)$, and the width grows with $x_{nl}^2$.

In [ ]:
for order, (x, angular_order, n) in enumerate(sphere_model.roots, start=1):
    width = sphere_model.calculate_width(order=order)[0]
    print(f'order {order:2d}: x = {x:6.3f}, l = {angular_order}, n = {n}, HWHM = {width:.4f} meV')

## EISF and QISF as a function of Q

At small $QR$ almost all intensity is elastic. As $QR$ grows, the weight moves into the Lorentzians, first into the one with the smallest root ($l = 1$). The EISF and all QISF together sum to one, minus the weight of the terms that are left out.

In [ ]:
Q_fine = np.linspace(0.01, 1.5, 200)

plt.figure()
plt.plot(Q_fine, sphere_model.calculate_EISF(Q_fine), color='k', label='EISF')
for order in range(1, 5):
    _, angular_order, n = sphere_model.roots[order - 1]
    plt.plot(
        Q_fine,
        sphere_model.calculate_QISF_order(Q_fine, order=order),
        label=f'QISF of order {order} (l = {angular_order}, n = {n})',
    )
plt.xlabel('Q (Å$^{-1}$)')
plt.ylabel('Weight')
plt.title('EISF and QISF for R = 1 Å')
plt.legend()
plt.show()

## Truncation

Leaving out terms loses intensity at large $QR$. A warning is raised when more than $10^{-3}$ of the weight is lost at any of the model's $Q$ values, and `calculate_missing_weight` returns the weight that is left out. With 10 terms this starts at $QR \approx 1.8$, and with all 98 at $QR \approx 9$. A model without $Q$ values builds no components, so it is quick to create for this check.

In [ ]:
QR = np.linspace(0.01, 12, 300)

plt.figure()
for number_of_lorentzians in [10, 30, 98]:
    model = edyn.DiffusionInSphere(radius=1.0, number_of_lorentzians=number_of_lorentzians)
    plt.semilogy(
        QR,
        np.maximum(model.calculate_missing_weight(QR), 1e-16),
        label=f'{number_of_lorentzians} terms',
    )
plt.axhline(1e-3, color='k', linestyle='dashed', label='warning threshold')
plt.xlabel('QR')
plt.ylabel('Missing weight')
plt.title('Weight left out by the truncated sum')
plt.legend()
plt.show()

A larger sphere reaches the threshold at smaller $Q$. With $R = 3$ Å and the default 10 terms, $Q$ up to 2 Å$^{-1}$ is already too far:

In [ ]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    large_sphere = edyn.DiffusionInSphere(radius=3.0, Q=np.linspace(0.3, 2, 4))
for warning in caught:
    print(warning.message)

print('Missing weight per Q:', np.round(large_sphere.calculate_missing_weight(), 4))

More terms remove the warning but make every update of the model slower. The spherical Bessel functions are faster with the optional numba dependency, `pip install easydynamics[fast]`; without it SciPy gives the same values.

## References

1. F. Volino and A. J. Dianoux, *Mol. Phys.* **41**, 271 (1980), [doi:10.1080/00268978000102761](https://doi.org/10.1080/00268978000102761).

2. D. J. Bicout, *Phys. Rev. E* **62**, 261 (2000), [doi:10.1103/PhysRevE.62.261](https://doi.org/10.1103/PhysRevE.62.261).

3. Mantid, [`InelasticDiffSphere`](https://github.com/mantidproject/mantid/blob/6ec66a6f524a7ec9d4258351fe3001c66c63dbe5/Framework/CurveFitting/src/Functions/InelasticDiffSphere.cpp#L53-L69): the table of the 98 roots.